# Phase 2: Citation Knowledge Graphs & Cartography
**Nexus Scholar Interactive Research Suite**

This notebook demonstrates how to construct directed citation and co-citation knowledge networks from screening results, compute normalized PageRank centrality scores, and render interactive PyVis visual graph maps.

In [ ]:
import asyncio
import json
from pathlib import Path
import networkx as nx
import pandas as pd
from IPython.display import display, HTML, IFrame
from scholar_graph.builder import CitationGraphBuilder
from scholar_graph.visualizer import GraphVisualizer

## 1. Load Included Literature & Build Citation Topology

In [ ]:
inc_file = Path("literature/included.json")
dois = []
if inc_file.exists():
    papers = json.loads(inc_file.read_text(encoding="utf-8"))
    dois = [p.get("doi") for p in papers if p.get("doi")]
else:
    # Sample seed DOIs
    dois = ["10.5555/3295222.3295349", "10.18653/v1/N19-1423"]

builder = CitationGraphBuilder()
G = asyncio.run(builder.build_graph(dois))
print(f"Graph Constructed: {G.number_of_nodes()} Nodes, {G.number_of_edges()} Edges")

## 2. Compute PageRank Centrality & Community Detection

In [ ]:
pr_scores = CitationGraphBuilder.compute_pagerank(G)
df_pr = pd.DataFrame([
    {
        "DOI / Node": node,
        "Title": data.get("title", "Unknown"),
        "Citations": data.get("citation_count", 0),
        "PageRank Score": round(pr_scores.get(node, 0.0), 5)
    }
    for node, data in G.nodes(data=True)
]).sort_values(by="PageRank Score", ascending=False)

display(df_pr.head(10))

## 3. Render Interactive PyVis HTML Map

In [ ]:
html_out = Path("literature/knowledge_graph.html")
vis = GraphVisualizer(str(html_out))
vis.generate_html(G)
print(f"Saved interactive PyVis canvas to {html_out}")

# Display iframe inline in notebook
IFrame(src=str(html_out), width="100%", height="600px")